# MCTNet vs GatedMCTNet — Comparaison Partie 3
## Arkansas · California
Entraîne MCTNet (baseline) et GatedMCTNet (Partie 3) sur les mêmes données
et affiche le tableau comparatif final.

## Cellule 0 — Montage Drive & vérification GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

import os, sys

NPY_DIR  = '/content/drive/MyDrive/Crop Classification/partie1'
CODE_DIR = '/content/drive/MyDrive/Crop Classification/src'

sys.path.insert(0, os.path.dirname(CODE_DIR))

print('\nVerification des fichiers .npy :')
all_ok = True
for region in ['Arkansas', 'California']:
    for split in ['train', 'val', 'test']:
        for suffix in ['input1', 'input2', 'labels']:
            f = f'{NPY_DIR}/{region}_{split}_{suffix}.npy'
            ok = os.path.exists(f)
            if not ok:
                all_ok = False
            print(f'  {"OK" if ok else "MANQUANT"}  {region}_{split}_{suffix}.npy')
print('\nTous les fichiers presents.' if all_ok else '\nFICHIERS MANQUANTS — verifier Drive.')

## Cellule 1 — Imports

In [ ]:
import os, sys, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
import matplotlib.pyplot as plt

from src.mctnet import MCTNet, GatedMCTNet

print('PyTorch :', torch.__version__)
print('Device  :', 'cuda' if torch.cuda.is_available() else 'cpu')
print('MCTNet et GatedMCTNet importes.')

## Cellule 2 — Configuration

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

CONFIG = {
    'lr':           0.001,
    'weight_decay': 1e-4,
    'batch_size':   32,
    'epochs':       200,
    'n_head':       5,
    'kernel_size':  3,
    'dropout':      0.1,
    'print_every':  10,
    'patience':     20,
}

N_CLASSES = {'Arkansas': 5, 'California': 6}

CLASS_NAMES = {
    'Arkansas':   ['Corn', 'Cotton', 'Rice', 'Soybeans', 'Others'],
    'California': ['Rice', 'Alfalfa', 'Grapes', 'Almonds', 'Pistachios', 'Others'],
}

all_results = {}
print('Configuration chargee. Seed=42.')

## Cellule 3 — Dataset & fonctions d'entraînement

In [ ]:
class CropDataset(Dataset):
    def __init__(self, data_dir, region, split):
        prefix    = os.path.join(data_dir, f'{region}_{split}')
        self.X    = torch.from_numpy(np.load(f'{prefix}_input1.npy')).float()
        self.mask = torch.from_numpy(np.load(f'{prefix}_input2.npy')).float()
        self.y    = torch.from_numpy(np.load(f'{prefix}_labels.npy')).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X, mask, y in loader:
        X, mask, y = X.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X, mask), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for X, mask, y in loader:
            X, mask, y = X.to(device), mask.to(device), y.to(device)
            logits = model(X, mask)
            total_loss += criterion(logits, y).item() * len(y)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    return (
        total_loss / len(loader.dataset),
        accuracy_score(all_labels, all_preds),
        cohen_kappa_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average='macro', zero_division=0),
    )

def run_training(model_name, region, config, device):
    model_cls = GatedMCTNet if model_name == 'gated' else MCTNet
    n_classes = N_CLASSES[region]
    save_path = f'best_{model_name}_{region}.pth'

    print('=' * 60)
    print(f'  {model_name.upper()} -- {region}  ({n_classes} classes)')
    print('=' * 60)

    train_set    = CropDataset(NPY_DIR, region, 'train')
    val_set      = CropDataset(NPY_DIR, region, 'val')
    test_set     = CropDataset(NPY_DIR, region, 'test')
    train_loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=config['batch_size'], shuffle=False)
    test_loader  = DataLoader(test_set,  batch_size=config['batch_size'], shuffle=False)
    print(f'  Donnees : {len(train_set)} train | {len(val_set)} val | {len(test_set)} test')

    torch.manual_seed(42)
    model = model_cls(
        n_classes=n_classes,
        n_head=config['n_head'],
        kernel_size=config['kernel_size'],
        dropout=config['dropout'],
    ).to(device)
    print(f'  Parametres : {sum(p.numel() for p in model.parameters()):,}')

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config['lr'], weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    best_f1, best_epoch, epochs_no_impr = 0.0, 0, 0
    history = {'train_loss': [], 'val_loss': [], 'val_oa': [], 'val_kappa': [], 'val_f1': []}
    t0 = time.time()

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_oa, val_kappa, val_f1 = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_oa'].append(val_oa)
        history['val_kappa'].append(val_kappa)
        history['val_f1'].append(val_f1)

        if val_f1 > best_f1:
            best_f1, best_epoch, epochs_no_impr = val_f1, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_impr += 1

        if epoch % config['print_every'] == 0 or epoch == 1:
            print(f'  Ep {epoch:3d}/{config["epochs"]} | train={train_loss:.4f} | val={val_loss:.4f} | OA={val_oa:.4f} | F1={val_f1:.4f} | best={best_f1:.4f}(ep{best_epoch}) | {time.time()-t0:.0f}s')

        if epochs_no_impr >= config['patience']:
            print(f'  Early stopping ep {epoch} (best ep={best_epoch})')
            break

    model.load_state_dict(torch.load(save_path, map_location=device))
    _, test_oa, test_kappa, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f'\n  TEST -- OA={test_oa:.4f} | Kappa={test_kappa:.4f} | F1={test_f1:.4f}')

    return history, {'OA': test_oa, 'Kappa': test_kappa, 'F1': test_f1}, save_path

print('Fonctions chargees.')

## Cellule 4 — Entraînement MCTNet (baseline)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
t_global = time.time()
all_histories = {}

for region in ['Arkansas', 'California']:
    history, metrics, path = run_training('mctnet', region, CONFIG, device)
    all_results.setdefault('mctnet', {})[region] = metrics
    all_histories.setdefault('mctnet', {})[region] = history

print(f'\nMCTNet termine en {(time.time()-t_global)/60:.1f} min')

## Cellule 5 — Entraînement GatedMCTNet (Partie 3)

In [ ]:
t_global = time.time()

for region in ['Arkansas', 'California']:
    history, metrics, path = run_training('gated', region, CONFIG, device)
    all_results.setdefault('gated', {})[region] = metrics
    all_histories.setdefault('gated', {})[region] = history

print(f'\nGatedMCTNet termine en {(time.time()-t_global)/60:.1f} min')

## Cellule 6 — Tableau comparatif final

In [ ]:
print('=' * 65)
print('  COMPARAISON MCTNet vs GatedMCTNet')
print('=' * 65)
print(f'  {"Modele":<14} {"Region":<12} {"OA":>8} {"Kappa":>8} {"F1":>8}')
print('-' * 65)
for model_name in ['mctnet', 'gated']:
    for region in ['Arkansas', 'California']:
        r = all_results.get(model_name, {}).get(region)
        if r:
            print(f'  {model_name:<14} {region:<12} {r["OA"]:>8.4f} {r["Kappa"]:>8.4f} {r["F1"]:>8.4f}')
print('=' * 65)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#F8F9FA')
fig.suptitle('Courbes apprentissage — MCTNet vs GatedMCTNet', fontsize=14, fontweight='bold')

COLORS = {'mctnet': '#3498DB', 'gated': '#E74C3C'}
LABELS = {'mctnet': 'MCTNet', 'gated': 'GatedMCTNet'}
regions = ['Arkansas', 'California']

for row, region in enumerate(regions):
    for model_name in ['mctnet', 'gated']:
        hist  = all_histories.get(model_name, {}).get(region)
        if hist is None:
            continue
        ep    = range(1, len(hist['train_loss']) + 1)
        color = COLORS[model_name]
        label = LABELS[model_name]

        axes[row][0].plot(ep, hist['train_loss'], color=color, label=f'{label} train', linewidth=1.5)
        axes[row][0].plot(ep, hist['val_loss'],   color=color, label=f'{label} val',   linewidth=1.5, linestyle='--')
        axes[row][1].plot(ep, hist['val_oa'],     color=color, label=label, linewidth=1.5)
        axes[row][2].plot(ep, hist['val_f1'],     color=color, label=label, linewidth=1.5)

    axes[row][0].set_title(f'{region} — Loss');        axes[row][0].legend(fontsize=7); axes[row][0].grid(alpha=0.3)
    axes[row][1].set_title(f'{region} — OA (val)');   axes[row][1].set_ylim(0, 1);     axes[row][1].legend(); axes[row][1].grid(alpha=0.3)
    axes[row][2].set_title(f'{region} — F1 (val)');   axes[row][2].set_ylim(0, 1);     axes[row][2].legend(); axes[row][2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('courbes_comparaison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : courbes_comparaison.png')

## Cellule 7 — Courbes d'apprentissage